<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_CRVG_Non_Securitization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Setup and Initial Portfolio
# -----------------------------------------------------------------------------
# This cell loads the initial portfolio data and regulatory parameters. The output displays
# the starting positions, which corresponds to Step 1 in the report.

# Import necessary libraries
import pandas as pd
import numpy as np

# --- Initial Portfolio Data for CSR Non-Securitisation ---
portfolio_data = [
    {'position_id': 1, 'bucket': 5, 'issuer': 'Issuer 1', 'tenor_str': '5Y', 'gross_sensitivity': 269},
    {'position_id': 2, 'bucket': 5, 'issuer': 'Issuer 1', 'tenor_str': '5Y', 'gross_sensitivity': 309},
    {'position_id': 3, 'bucket': 5, 'issuer': 'Issuer 2', 'tenor_str': '5Y', 'gross_sensitivity': -302},
    {'position_id': 4, 'bucket': 5, 'issuer': 'Issuer 2', 'tenor_str': '5Y', 'gross_sensitivity': -44},
]

# --- Regulatory Parameters for Vega CSR Non-Sec ---
TENOR_TO_YEARS = {'5Y': 5}
VEGA_RISK_WEIGHT = 1.00 # As per Art. 325ax
# As per BCBS MAR21 FAQ on 21.94, for Vega CSR non-sec, the delta component only considers the 'name'.
# From Art. 325ai, rho(name) for different issuers in this bucket is 35%.
RHO_DELTA_DIFFERENT_ISSUER = 0.35
ALPHA = 0.01 # Parameter for option maturity correlation

# Create the initial DataFrame
df = pd.DataFrame(portfolio_data)

print("--- Step 1 & 2: Initial Portfolio, Risk Factors, and Gross Sensitivities ---")
print("The calculation begins with the initial portfolio positions. Each unique combination")
print("of 'issuer' and 'tenor_str' represents a distinct risk factor.")
print("\n" + df[['position_id', 'bucket', 'issuer', 'tenor_str', 'gross_sensitivity']].to_string(index=False))

--- Step 1 & 2: Initial Portfolio, Risk Factors, and Gross Sensitivities ---
The calculation begins with the initial portfolio positions. Each unique combination
of 'issuer' and 'tenor_str' represents a distinct risk factor.

 position_id  bucket   issuer tenor_str  gross_sensitivity
           1       5 Issuer 1        5Y                269
           2       5 Issuer 1        5Y                309
           3       5 Issuer 2        5Y               -302
           4       5 Issuer 2        5Y                -44


In [2]:
# Cell 2: Step 3 - Net Sensitivities
# -----------------------------------------------------------------------------
# As per Article 325f(5), sensitivities for identical risk factors are netted.
# We group by the risk factor components (issuer, tenor) and sum the sensitivities.
df_net = df.groupby(['bucket', 'issuer', 'tenor_str']).agg(
    net_sensitivity=('gross_sensitivity', 'sum')
).reset_index()
df_net['tenor_years'] = df_net['tenor_str'].map(TENOR_TO_YEARS)

print("\n\n--- Step 3: Net Sensitivities ---")
print("Gross sensitivities for each unique risk factor are summed to get the net sensitivity.")
print("\n" + df_net[['issuer', 'tenor_str', 'net_sensitivity']].to_string(index=False))



--- Step 3: Net Sensitivities ---
Gross sensitivities for each unique risk factor are summed to get the net sensitivity.

  issuer tenor_str  net_sensitivity
Issuer 1        5Y              578
Issuer 2        5Y             -346


In [3]:
# Cell 3: Step 4 - Weighted Sensitivities
# -----------------------------------------------------------------------------
# Each net sensitivity is multiplied by the regulatory risk weight (100% for Vega CSR).
df_weighted = df_net.copy()
df_weighted['weighted_sensitivity'] = df_weighted['net_sensitivity'] * VEGA_RISK_WEIGHT

# Calculate Sb for later steps
S_b = df_weighted['weighted_sensitivity'].sum()

print("\n\n--- Step 4: Weighted Sensitivities ---")
print("Net sensitivities are multiplied by the regulatory risk weight for the bucket.")
print("\n" + df_weighted[['issuer', 'tenor_str', 'net_sensitivity', 'weighted_sensitivity']].round(2).to_string(index=False))
print(f"\nSum of Weighted Sensitivities (S_5): {S_b:,.2f}")




--- Step 4: Weighted Sensitivities ---
Net sensitivities are multiplied by the regulatory risk weight for the bucket.

  issuer tenor_str  net_sensitivity  weighted_sensitivity
Issuer 1        5Y              578                 578.0
Issuer 2        5Y             -346                -346.0

Sum of Weighted Sensitivities (S_5): 232.00


In [4]:
# Cell 4: Step 5 - Intra-Bucket Correlation Coefficients
# -----------------------------------------------------------------------------
# Per Article 325ay, Vega correlation is a product of a delta component and an option maturity component.
# For CSR non-sec, the delta component (rho_delta_comp) simplifies to only rho(name).
positions = df_weighted.to_dict('records')
correlation_data = []
intra_corrs_medium = {}

pos1, pos2 = positions[0], positions[1] # There is only one pair of risk factors

pair_name = f"{pos1['issuer']}-{pos1['tenor_str']} vs {pos2['issuer']}-{pos2['tenor_str']}"
pair_key = tuple(sorted((f"{pos1['issuer']}_{pos1['tenor_str']}", f"{pos2['issuer']}_{pos2['tenor_str']}")))

# 1. Delta Component: Based only on the name as per BCBS MAR21.94 FAQ
rho_delta_comp = 1.0 if pos1['issuer'] == pos2['issuer'] else RHO_DELTA_DIFFERENT_ISSUER

# 2. Option Maturity Component
if pos1['tenor_years'] == pos2['tenor_years']:
    rho_maturity_comp = 1.0
else:
    t1, t2 = pos1['tenor_years'], pos2['tenor_years']
    rho_maturity_comp = np.exp(-ALPHA * abs(t1 - t2) / min(t1, t2))

# 3. Final Vega Correlation
rho_kl = rho_delta_comp * rho_maturity_comp
intra_corrs_medium[pair_key] = rho_kl
correlation_data.append([pair_name, f"{rho_delta_comp:.2%}", f"{rho_maturity_comp:.2%}", f"{rho_kl:.2%}"])

df_correlations = pd.DataFrame(correlation_data, columns=["Pair of Risk Factors", "Delta Component", "Maturity Component", "Final Correlation"])

print("\n\n--- Step 5: Intra-Bucket Correlation Coefficients (Medium Scenario) ---")
print("Correlations are determined based on issuer similarity and option tenor similarity.")
print("\n" + df_correlations.to_string(index=False))



--- Step 5: Intra-Bucket Correlation Coefficients (Medium Scenario) ---
Correlations are determined based on issuer similarity and option tenor similarity.

      Pair of Risk Factors Delta Component Maturity Component Final Correlation
Issuer 1-5Y vs Issuer 2-5Y          35.00%            100.00%            35.00%


In [5]:
# Cell 5: Step 7 - Intra-Bucket Aggregation (Medium Scenario)
# -----------------------------------------------------------------------------
# Calculate the bucket-specific capital charge (K_b) for the Medium Scenario.
positions_agg = df_weighted.to_dict('records')
sum_ws_sq = np.sum(df_weighted['weighted_sensitivity']**2)
cross_term = 0

p1, p2 = positions_agg[0], positions_agg[1]
key_part1 = f"{p1['issuer']}_{p1['tenor_str']}"
key_part2 = f"{p2['issuer']}_{p2['tenor_str']}"
pair_key = tuple(sorted((key_part1, key_part2)))
rho_kl_medium = intra_corrs_medium.get(pair_key, 0)
cross_term += 2 * rho_kl_medium * p1['weighted_sensitivity'] * p2['weighted_sensitivity']

K_b_medium = np.sqrt(max(0, sum_ws_sq + cross_term))

print("\n\n--- Step 7: Intra-Bucket Aggregation (Medium Scenario) ---")
print("Weighted sensitivities are aggregated using the specified correlations.")
print(f"\n1. Sum of Squares (Σ WS_k^2): {sum_ws_sq:,.2f}")
print(f"2. Sum of Cross-Products (Σ ρ_kl * WS_k * WS_l): {cross_term:,.2f}")
print("----------------------------------------------------------")
print(f"Bucket 5 Capital (K_5): {K_b_medium:,.2f}")



--- Step 7: Intra-Bucket Aggregation (Medium Scenario) ---
Weighted sensitivities are aggregated using the specified correlations.

1. Sum of Squares (Σ WS_k^2): 453,800.00
2. Sum of Cross-Products (Σ ρ_kl * WS_k * WS_l): -139,991.60
----------------------------------------------------------
Bucket 5 Capital (K_5): 560.19


In [6]:
# Cell 6: Step 9 - High and Low Correlation Scenarios
# -----------------------------------------------------------------------------
# Recalculate the total capital for High and Low correlation scenarios.
def calculate_capital_for_scenario(df_to_calc, medium_corr, scenario):
    if scenario == 'High':
        stressed_corr = min(medium_corr * 1.25, 1.0)
    elif scenario == 'Low':
        stressed_corr = max(2 * medium_corr - 1.0, 0.75 * medium_corr)
    else: # Medium
        stressed_corr = medium_corr

    sum_ws_sq = np.sum(df_to_calc['weighted_sensitivity']**2)
    positions = df_to_calc.to_dict('records')
    p1, p2 = positions[0], positions[1]

    cross_term = 2 * stressed_corr * p1['weighted_sensitivity'] * p2['weighted_sensitivity']

    return np.sqrt(max(0, sum_ws_sq + cross_term))

K_b_high = calculate_capital_for_scenario(df_weighted, rho_kl_medium, 'High')
K_b_low = calculate_capital_for_scenario(df_weighted, rho_kl_medium, 'Low')

scenario_data = {
    'Scenario': ['Medium Correlation', 'High Correlation', 'Low Correlation'],
    'Credit Vega Capital Requirement': [K_b_medium, K_b_high, K_b_low]
}
df_scenarios = pd.DataFrame(scenario_data)

print("\n\n--- Step 9: Correlation Scenarios ---")
print("The capital is recalculated under stressed correlation assumptions.")
print("\nResulting Capital per Scenario:")
print(df_scenarios.round(2).to_string(index=False))



--- Step 9: Correlation Scenarios ---
The capital is recalculated under stressed correlation assumptions.

Resulting Capital per Scenario:
          Scenario  Credit Vega Capital Requirement
Medium Correlation                           560.19
  High Correlation                           528.03
   Low Correlation                           590.60


In [7]:
# Cell 7: Step 10 - Final Charge Calculation
# -----------------------------------------------------------------------------
# The final charge is the maximum of the three scenarios.
final_charge = df_scenarios['Credit Vega Capital Requirement'].max()
winning_scenario = df_scenarios.loc[df_scenarios['Credit Vega Capital Requirement'].idxmax()]['Scenario']

print("\n\n--- Step 10: Final Charge Calculation ---")
print("The final requirement is the maximum of the three scenarios.")
print("\n-------------------------------------------------")
print(f" Final Credit Vega Capital Requirement: {final_charge:,.2f}")
print(f" (Driven by the {winning_scenario})")
print("-------------------------------------------------")



--- Step 10: Final Charge Calculation ---
The final requirement is the maximum of the three scenarios.

-------------------------------------------------
 Final Credit Vega Capital Requirement: 590.60
 (Driven by the Low Correlation)
-------------------------------------------------
